In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_dir = "data/"

In [3]:
train_features = pd.read_csv(f'{data_dir}train_hh_features.csv')
train_features

,hhid,com,weight,strata,utl_exp_ppp17,male,hsize,num_children5,num_children10,num_children18,...,consumed4200,consumed4300,consumed4400,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,survey_id
0,100001,1,75,4,594.80627,Female,1,0,0,0,...,Yes,No,No,No,Yes,Yes,Yes,Yes,No,100000
1,100002,1,150,4,1676.27230,Female,2,0,0,0,...,Yes,No,No,No,No,Yes,Yes,No,No,100000
2,100003,1,375,4,506.93719,Male,5,0,0,2,...,Yes,Yes,No,Yes,Yes,Yes,Yes,No,Yes,100000
3,100004,1,375,4,824.61786,Male,5,0,0,1,...,No,Yes,No,No,No,Yes,Yes,No,No,100000
4,100005,1,525,4,351.47644,Male,7,1,0,0,...,Yes,No,No,Yes,No,Yes,Yes,Yes,No,100000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104229,337458,1,616,7,143.58849,Female,4,2,1,0,...,Yes,No,No,No,No,Yes,Yes,No,Yes,300000
104230,337459,1,616,7,268.44803,Female,4,0,0,1,...,Yes,No,No,No,No,Yes,No,No,No,300000
104231,337460,1,1540,7,243.47612,Male,10,2,3,2,...,Yes,No,No,Yes,No,Yes,Yes,No,No,300000
104232,337461,1,1232,7,337.12079,Male,8,0,0,3,...,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,300000


In [4]:
train_consumption =pd.read_csv(f'{data_dir}train_hh_gt.csv')
train_consumption

,survey_id,hhid,cons_ppp17
0,100000,100001,25.258402
1,100000,100002,16.996706
2,100000,100003,13.671848
3,100000,100004,7.189475
4,100000,100005,12.308855
...,...,...,...
104229,300000,337458,2.830888
104230,300000,337459,3.144309
104231,300000,337460,3.319158
104232,300000,337461,6.088739


In [5]:
feature_desc = pd.read_csv(f'{data_dir}feature_descriptions.csv')
feature_desc

,Variable name,Storage type,Variable label
0,sample,str8,Data Type
1,hhid,int,Household unique identifier
2,com,byte,Identifier of household member
3,weight,int,Household sampling weight
4,strata,byte,Stratification variable
...,...,...,...
103,_pline75,float,Poverty line: 75th percentile of welfare distr...
104,_pline80,float,Poverty line: 80th percentile of welfare distr...
105,_pline85,float,Poverty line: 85th percentile of welfare distr...
106,_pline90,float,Poverty line: 90th percentile of welfare distr...


In [6]:
mask=feature_desc['Variable name']=='cons_ppp17'#target varible
filtered_rows = feature_desc[mask]
print(filtered_rows)

  Variable name Storage type  \
5    cons_ppp17        float   

                                      Variable label  
5  Target variable: Per-capita daily expenditure ...  


In [7]:
rates = pd.read_csv(f'{data_dir}train_rates_gt.csv')
rates

,survey_id,pct_hh_below_3.17,pct_hh_below_3.94,pct_hh_below_4.60,pct_hh_below_5.26,pct_hh_below_5.88,pct_hh_below_6.47,pct_hh_below_7.06,pct_hh_below_7.70,pct_hh_below_8.40,pct_hh_below_9.13,pct_hh_below_9.87,pct_hh_below_10.70,pct_hh_below_11.62,pct_hh_below_12.69,pct_hh_below_14.03,pct_hh_below_15.64,pct_hh_below_17.76,pct_hh_below_20.99,pct_hh_below_27.37
0,100000,0.067364,0.118927,0.169905,0.221865,0.271564,0.319585,0.366329,0.419816,0.471454,0.523798,0.574413,0.623091,0.671263,0.721329,0.773303,0.819770,0.865121,0.909075,0.954239
1,200000,0.059326,0.111560,0.159023,0.211754,0.263100,0.311758,0.356914,0.407631,0.463443,0.512931,0.559361,0.609337,0.659291,0.708043,0.760932,0.809045,0.860350,0.906385,0.952805
2,300000,0.049803,0.100381,0.149502,0.200144,0.250192,0.300211,0.349596,0.399930,0.449845,0.499930,0.550082,0.599926,0.650088,0.699617,0.750341,0.800111,0.850081,0.899974,0.949988


In [8]:
train_data = train_features.copy()

In [9]:
train_data['dependency_ratio'] = (train_data['num_children5'] + train_data['num_children10'] + train_data['num_elderly']) / train_data['hsize']

In [10]:
columns_to_drop = ['utl_exp_ppp17', 'weight', 'hhid', 'survey_id', 'strata', 'com']
train_data = train_data.drop(columns=columns_to_drop)

In [11]:
train_data['worker_share'] = train_data['sworkershh'] / train_data['hsize']

In [12]:
train_data['housing_index'] = (
    train_data['water'] + 
    train_data['toilet'] + 
    train_data['sewer'] + 
    train_data['elect']
)

In [13]:
train_data['log_hsize'] = np.log1p(train_data['hsize'])

In [14]:
train_data[['urban', 'educ_max']]

,urban,educ_max
0,Urban,Incomplete Secondary Education
1,Urban,Complete Tertiary Education
2,Urban,Incomplete Tertiary Education
3,Urban,Incomplete Tertiary Education
4,Urban,Complete Tertiary Education
...,...,...
104229,Rural,Complete Primary Education
104230,Rural,Complete Secondary Education
104231,Rural,Complete Secondary Education
104232,Rural,Complete Tertiary Education


In [15]:
education_map = {
    'Never attended': 0,
    'Incomplete Primary Education': 1,
    'Complete Primary Education': 2,
    'Incomplete Secondary Education': 3,
    'Complete Secondary Education': 4,
    'Incomplete Tertiary Education': 5,
    'Complete Tertiary Education': 6
}
urban_map = {
    'Urban': 1,
    'Rural': 0
}
train_data['educ_max_numeric'] = train_data['educ_max'].map(education_map).fillna(0)
train_data['urban_numeric'] = train_data['urban'].map(urban_map)
train_data = train_data.drop(columns=['educ_max','urban'])
train_data['urban_educ_interaction']= train_data['urban_numeric']*train_data['educ_max_numeric']

In [16]:
consumed_cols = [ 'consumed100',
       'consumed200', 'consumed300', 'consumed400', 'consumed500',
       'consumed600', 'consumed700', 'consumed800', 'consumed900',
       'consumed1000', 'consumed1100', 'consumed1200', 'consumed1300',
       'consumed1400', 'consumed1500', 'consumed1600', 'consumed1700',
       'consumed1800', 'consumed1900', 'consumed2000', 'consumed2100',
       'consumed2200', 'consumed2300', 'consumed2400', 'consumed2500',
       'consumed2600', 'consumed2700', 'consumed2800', 'consumed2900',
       'consumed3000', 'consumed3100', 'consumed3200', 'consumed3300',
       'consumed3400', 'consumed3500', 'consumed3600', 'consumed3700',
       'consumed3800', 'consumed3900', 'consumed4000', 'consumed4100',
       'consumed4200', 'consumed4300', 'consumed4400', 'consumed4500',
       'consumed4600', 'consumed4700', 'consumed4800', 'consumed4900',
       'consumed5000']

In [17]:
dummies_cols = ['male','owner','water','toilet','sewer','elect','dweltyp','sector1d']

In [18]:
print(train_data['consumed1400'].unique())

['No' 'Yes' nan]


In [19]:
#Ranking responses based on their meaning, so as to encode their significance 
water_source_map = {
    'Surface water': 0,
    'Other': 1,
    'Protected spring': 2,
    'Protected dug well': 2,
    'Public tap or standpipe': 3,
    'Tanker-truck': 3,
    'Piped water to yard/plot': 4,
    'Piped water into dwelling': 5
}

sanitation_map = {
    'No facilities or bush or field': 0,
    'Other': 1,
    'Pit latrine': 1,
    'Pit latrine with slab': 2,
    'A septic tank': 3,
    'A piped sewer system': 4
}

employed_map = {
    'Not employed': 0,
    'Employed': 1,
    'nan':0
}

nonagric_map = {
    'Yes': 1,
    'No': 0
}

binary_map = {'yes': 1, 'no': 0}

In [20]:
#Create a score for housing index situation
def parse_housing_index(text):
    if pd.isna(text):
        return 0
    no_access_count =text.count("No access")
    score = 4 - no_access_count
    return max(0, score)

In [21]:
train_data['housing_index_score'] =train_data['housing_index'].apply(parse_housing_index)
train_data = train_data.drop(columns=['housing_index'], errors='ignore')

In [22]:
train_data['water_source_numeric'] = train_data['water_source'].map(water_source_map).fillna(1)
train_data['sanitation_source_numeric']= train_data['sanitation_source'].map(sanitation_map).fillna(1)
train_data['employed_numeric'] = train_data['employed'].map(employed_map).fillna(0)
train_data['nonagric_numeric'] =train_data['any_nonagric'].map(nonagric_map).fillna(0)
train_data[consumed_cols] = train_data[consumed_cols].replace(binary_map)

train_data = train_data.drop(columns=['water_source','sanitation_source','employed','any_nonagric'])
train_data = train_data.drop(columns=consumed_cols)

In [23]:
train_data = pd.get_dummies(train_data, columns=dummies_cols, dummy_na=True, dtype=int)

In [24]:
train_data.dtypes

hsize                                                    int64
num_children5                                            int64
num_children10                                           int64
num_children18                                           int64
age                                                      int64
                                                         ...  
sector1d_Public administration and defence               int64
sector1d_Real estate, renting and business activities    int64
sector1d_Transport, storage and communications           int64
sector1d_Wholesale and retail trade                      int64
sector1d_nan                                             int64
Length: 70, dtype: object

In [25]:
train_data['utl_exp_ppp17'] = train_features['utl_exp_ppp17']
train_data['weight'] = train_features['weight']
train_data

,hsize,num_children5,num_children10,num_children18,age,num_adult_female,num_adult_male,num_elderly,sworkershh,share_secondary,...,sector1d_Manufacturing,sector1d_Mining and quarrying,"sector1d_Other community, social and personal service activities",sector1d_Public administration and defence,"sector1d_Real estate, renting and business activities","sector1d_Transport, storage and communications",sector1d_Wholesale and retail trade,sector1d_nan,utl_exp_ppp17,weight
0,1,0,0,0,75,0,0,1,0.000000,0.000000,...,0,0,0,0,0,0,0,1,594.80627,75
1,2,0,0,0,61,1,0,1,0.000000,0.000000,...,0,0,0,0,0,0,0,1,1676.27230,150
2,5,0,0,2,49,1,2,0,0.666667,0.333333,...,0,0,0,0,0,1,0,0,506.93719,375
3,5,0,0,1,58,1,3,0,1.250000,0.250000,...,0,0,0,1,0,0,0,0,824.61786,375
4,7,1,0,0,57,2,3,1,0.666667,0.333333,...,0,0,0,0,0,0,0,0,351.47644,525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104229,4,2,1,0,33,1,0,0,1.000000,0.000000,...,0,0,0,0,0,0,0,0,143.58849,616
104230,4,0,0,1,48,3,0,0,0.000000,0.333333,...,0,0,0,0,0,0,0,1,268.44803,616
104231,10,2,3,2,54,2,1,0,1.000000,0.333333,...,0,0,0,0,0,0,0,1,243.47612,1540
104232,8,0,0,3,49,1,4,0,1.000000,0.200000,...,0,0,0,0,0,0,1,0,337.12079,1232


In [26]:
train_data.to_csv('data/train_processed.csv', index=False)